In [1]:
import asyncio
import os
from typing import List, Dict, Any, Tuple
from dotenv import load_dotenv
from model_service import run_ocr_inference, get_all_ocr_model_names
from ocr_evaluation import evaluate_invoice_ocr

load_dotenv()

async def process_single_ocr_test(image_path: str, ground_truth: str, model_names: List[str]) -> Dict[str, Any]:
    """
    Process a single OCR test case and return results for all models
    
    Args:
        image_path: Path to the invoice image file (local path)
        ground_truth: Ground truth text from the invoice
        model_names: List of OCR model names to compare
        
    Returns:
        Dictionary with evaluation results for all models
    """
    try:
        # Get OCR outputs from all models concurrently
        ocr_responses = await asyncio.gather(
            *[run_ocr_inference(model_name, image_path) for model_name in model_names],
            return_exceptions=True
        )
        
        # Process responses and evaluations
        results = {
            "image_path": image_path,
            "model_names": model_names
        }
        
        model_scores = {}
        
        for model_name, response in zip(model_names, ocr_responses):
            # Handle exceptions
            if isinstance(response, Exception):
                print(f"Error with {model_name}: {str(response)}")
                response = ""
            
            # Evaluate response
            eval_result = evaluate_invoice_ocr(response, ground_truth)
            
            # Store metrics for each model
            model_key = model_name.replace(" ", "_")
            results[f"Field_Accuracy_{model_key}"] = eval_result["detailed_metrics"].get("field_accuracy", {}).get("score", 0.0)
            results[f"Line_Item_Quality_{model_key}"] = eval_result["detailed_metrics"].get("line_item_quality", {}).get("score", 0.0)
            results[f"Financial_Consistency_{model_key}"] = eval_result["detailed_metrics"].get("financial_consistency", {}).get("score", 0.0)
            results[f"Overall_Score_{model_key}"] = eval_result.get("overall_score", 0.0)
            results[f"Response_{model_key}"] = response[:500] if response else ""  # Truncate for display
            
            model_scores[model_name] = eval_result.get("overall_score", 0.0)
        
        # Determine winner (model with highest overall score)
        if model_scores:
            best_model = max(model_scores.items(), key=lambda x: x[1])
            results["winner"] = best_model[0]
            results["winner_score"] = best_model[1]
        else:
            results["winner"] = "None"
            results["winner_score"] = 0.0
        
        return results
        
    except Exception as e:
        print(f"Error processing image '{image_path}': {str(e)}")
        # Return error result with zeros for all models
        error_result = {
            "image_path": image_path,
            "model_names": model_names,
            "error": str(e),
            "winner": "Error"
        }
        for model_name in model_names:
            model_key = model_name.replace(" ", "_")
            error_result[f"Field_Accuracy_{model_key}"] = 0.0
            error_result[f"Line_Item_Quality_{model_key}"] = 0.0
            error_result[f"Financial_Consistency_{model_key}"] = 0.0
            error_result[f"Overall_Score_{model_key}"] = 0.0
        return error_result

async def process_ocr_tests(test_cases: List[Tuple[str, str]], model_names: List[str]) -> List[Dict[str, Any]]:
    """
    Process a list of OCR test cases with all models
    
    Args:
        test_cases: List of tuples (image_path, ground_truth_text)
        model_names: List of OCR model names to compare
        
    Returns:
        List of dictionaries with evaluation results for each test case
    """
    print(f"Comparing OCR models: {', '.join(model_names)}")
    print(f"Processing {len(test_cases)} test cases...")
    
    # Process all test cases
    results = []
    for i, (image_path, ground_truth) in enumerate(test_cases, 1):
        print(f"Processing test case {i}/{len(test_cases)}: {os.path.basename(image_path)}...")
        result = await process_single_ocr_test(image_path, ground_truth, model_names)
        results.append(result)
        
        # Print scores for all models
        score_parts = []
        for model in model_names:
            model_key = model.replace(" ", "_")
            score = result.get(f'Overall_Score_{model_key}', 0.0)
            score_parts.append(f"{model}: {score:.2f}")
        score_summary = ", ".join(score_parts)
        print(f"Scores: {score_summary}")
        print(f"Winner: {result.get('winner', 'N/A')}")
        print(f"Completed test case {i}/{len(test_cases)}")
    
    return results

def process_ocr_tests_sync(test_cases: List[Tuple[str, str]], model_names: List[str]) -> List[Dict[str, Any]]:
    """Synchronous wrapper for process_ocr_tests - works in Jupyter notebooks"""
    try:
        # Try to get the current event loop
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # If we're in a running event loop (like Jupyter), use nest_asyncio
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(process_ocr_tests(test_cases, model_names))
        else:
            # If no event loop is running, use asyncio.run
            return asyncio.run(process_ocr_tests(test_cases, model_names))
    except RuntimeError:
        # Fallback for Jupyter notebooks
        import nest_asyncio
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(process_ocr_tests(test_cases, model_names))

# Example usage:
if True:
    # Automatically load test images and ground truth texts from test folder
    from pathlib import Path
    
    test_folder = Path("test")
    images_folder = test_folder / "images"
    text_folder = test_folder / "text"
    
    ocr_test_cases = []
    
    if test_folder.exists() and images_folder.exists() and text_folder.exists():
        # Get all invoice images from the images subfolder
        image_files = sorted(images_folder.glob("invoice_*.png"))
        
        print(f"Found {len(image_files)} invoice images in {images_folder}")
        
        for img_path in image_files:
            # Find corresponding text file
            img_name = img_path.stem  # e.g., "invoice_0"
            txt_path = text_folder / f"{img_name}.txt"
            
            if txt_path.exists():
                # Read ground truth text
                with open(txt_path, 'r', encoding='utf-8') as f:
                    ground_truth = f.read().strip()
                
                ocr_test_cases.append((str(img_path.absolute()), ground_truth))
                print(f"Loaded: {img_path.name} with ground truth from {txt_path.name}")
            else:
                print(f"Warning: No ground truth found for {img_path.name} (expected {txt_path}), skipping...")
        
        print(f"\nTotal test cases loaded: {len(ocr_test_cases)}")
    else:
        print("Test folder structure not found.")
        if not test_folder.exists():
            print(f"  - Test folder '{test_folder}' does not exist")
        if not images_folder.exists():
            print(f"  - Images folder '{images_folder}' does not exist")
        if not text_folder.exists():
            print(f"  - Text folder '{text_folder}' does not exist")
        print("\nPlease provide OCR test cases manually.")
        ocr_test_cases = [
            # Format: (image_path, ground_truth_text)
            # Example:
            # ("test/images/invoice_0.png", "Invoice Number: INV-001\nSupplier: ABC Corp\n..."),
        ]
    
    # Get all available OCR models
    available_models = get_all_ocr_model_names()
    print(f"\nAvailable OCR models: {available_models}")
    
    if available_models and ocr_test_cases:
        # Process all test cases with all available models
        results = process_ocr_tests_sync(ocr_test_cases, available_models)
        
        # Print summary results
        print("\n=== SUMMARY RESULTS ===")
        for result in results:
            print(f"\nImage: {os.path.basename(result['image_path'])}")
            print(f"Winner: {result.get('winner', 'N/A')} (Score: {result.get('winner_score', 0.0):.2f})")
            for model in available_models:
                model_key = model.replace(" ", "_")
                print(f"  {model}: Overall={result.get(f'Overall_Score_{model_key}', 0.0):.2f}, "
                      f"Field={result.get(f'Field_Accuracy_{model_key}', 0.0):.2f}, "
                      f"LineItem={result.get(f'Line_Item_Quality_{model_key}', 0.0):.2f}, "
                      f"Financial={result.get(f'Financial_Consistency_{model_key}', 0.0):.2f}")
            if 'error' in result:
                print(f"  Error: {result['error']}")
    else:
        print("\nPlease provide OCR test cases and ensure models are available.")
        print("Test cases should be in format: [(image_path, ground_truth_text), ...]")
        if not available_models:
            print("No OCR models are currently available.")
        if not ocr_test_cases:
            print("No test cases were loaded. Check that test folder exists with images and text files.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 12-16 13:44:46 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Found 10 invoice images in test/images
Loaded: invoice_0.png with ground truth from invoice_0.txt
Loaded: invoice_1.png with ground truth from invoice_1.txt
Loaded: invoice_2.png with ground truth from invoice_2.txt
Loaded: invoice_3.png with ground truth from invoice_3.txt
Loaded: invoice_4.png with ground truth from invoice_4.txt
Loaded: invoice_5.png with ground truth from invoice_5.txt
Loaded: invoice_6.png with ground truth from invoice_6.txt
Loaded: invoice_7.png with ground truth from invoice_7.txt
Loaded: invoice_8.png with ground truth from invoice_8.txt
Loaded: invoice_9.png with ground truth from invoice_9.txt

Total test cases loaded: 10

Available OCR models: ['DeepSeek OCR', 'Qwen3-VL', 'Granite Docling', 'Chandra OCR']
Comparing OCR models: DeepSeek OCR, Qwen3-VL, Granite Docling, Chandra OCR
Processing 10 test cases...
Processing test case 1/10: invoice_0.png...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


...

Output()

Output()

Output()

Output()

Scores: DeepSeek OCR: 0.85, Qwen3-VL: 0.82, Granite Docling: 0.90, Chandra OCR: 0.82
Winner: Granite Docling
Completed test case 10/10

=== SUMMARY RESULTS ===

Image: invoice_0.png
Winner: Qwen3-VL (Score: 0.98)
  DeepSeek OCR: Overall=0.82, Field=0.46, LineItem=0.99, Financial=1.00
  Qwen3-VL: Overall=0.98, Field=0.95, LineItem=1.00, Financial=1.00
  Granite Docling: Overall=0.94, Field=0.81, LineItem=1.00, Financial=1.00
  Chandra OCR: Overall=0.78, Field=0.72, LineItem=0.66, Financial=0.94

Image: invoice_1.png
Winner: Qwen3-VL (Score: 0.99)
  DeepSeek OCR: Overall=0.82, Field=0.45, LineItem=1.00, Financial=1.00
  Qwen3-VL: Overall=0.99, Field=0.99, LineItem=0.99, Financial=1.00
  Granite Docling: Overall=0.95, Field=0.90, LineItem=0.96, Financial=1.00
  Chandra OCR: Overall=0.63, Field=0.74, LineItem=0.47, Financial=0.68

Image: invoice_2.png
Winner: Granite Docling (Score: 0.97)
  DeepSeek OCR: Overall=0.82, Field=0.49, LineItem=0.98, Financial=1.00
  Qwen3-VL: Overall=0.96, Fiel

In [2]:
import pandas as pd
import numpy as np
from typing import List, Dict, Any
import os

def create_results_table(results: List[Dict[str, Any]]) -> pd.DataFrame:
    """Create a formatted table from OCR evaluation results for all models"""
    # Create DataFrame from results
    df = pd.DataFrame(results)
    
    # Get model names from first result (assuming all results have same models)
    if len(results) > 0:
        model_names = results[0].get('model_names', [])
    else:
        model_names = []
    
    if not model_names:
        return pd.DataFrame()
    
    # Extract just filename from image path for display
    df['image_filename'] = df['image_path'].apply(lambda x: os.path.basename(x) if isinstance(x, str) else str(x))
    
    # Build column list dynamically for all models
    columns_to_select = ['image_filename', 'winner']
    
    # For each model, add its metric columns
    for model_name in model_names:
        model_key = model_name.replace(" ", "_")
        columns_to_select.extend([
            f'Field_Accuracy_{model_key}',
            f'Line_Item_Quality_{model_key}',
            f'Financial_Consistency_{model_key}',
            f'Overall_Score_{model_key}'
        ])
    
    # Round all numeric columns to 3 decimal places
    numeric_columns = [col for col in df.columns if any(metric in col for metric in ['Field_Accuracy', 'Line_Item_Quality', 'Financial_Consistency', 'Overall_Score'])]
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').round(3)
    
    # Select columns that exist
    available_columns = [col for col in columns_to_select if col in df.columns]
    final_df = df[available_columns].copy()
    
    # Rename columns for better readability
    rename_dict = {'image_filename': 'Image', 'winner': 'Winner'}
    for model_name in model_names:
        model_key = model_name.replace(" ", "_")
        rename_dict[f'Field_Accuracy_{model_key}'] = f'Field Acc ({model_name})'
        rename_dict[f'Line_Item_Quality_{model_key}'] = f'Line Item ({model_name})'
        rename_dict[f'Financial_Consistency_{model_key}'] = f'Financial ({model_name})'
        rename_dict[f'Overall_Score_{model_key}'] = f'Overall ({model_name})'
    
    final_df = final_df.rename(columns=rename_dict)
    
    return final_df

def display_results_table(results: List[Dict[str, Any]]):
    """Display the results table with formatting"""
    df = create_results_table(results)
    
    if df.empty:
        print("No results to display.")
        return df
    
    # Get model names
    model_names = results[0].get('model_names', []) if results else []
    
    # Display the table
    print("=== OCR EVALUATION RESULTS TABLE ===")
    print(df.to_string(index=False))
    
    # Print summary statistics
    print("\n=== SUMMARY STATISTICS ===")
    print(f"Total test cases: {len(df)}")
    
    # Count wins for each model
    win_counts = {}
    for model_name in model_names:
        wins = len(df[df['Winner'] == model_name])
        if wins > 0:
            win_counts[model_name] = wins
    
    if win_counts:
        print("\nWins per model:")
        for model_name, wins in sorted(win_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"  {model_name}: {wins}")
    
    # Average scores for each model
    print(f"\nAverage Overall Scores:")
    for model_name in model_names:
        model_key = model_name.replace(" ", "_")
        col_name = f'Overall ({model_name})'
        if col_name in df.columns:
            avg_score = df[col_name].mean()
            print(f"  {model_name}: {avg_score:.3f}")
    
    return df

# If you want to display in Jupyter with better formatting:
def display_formatted_table(results: List[Dict[str, Any]]):
    """Display formatted table with styling (for Jupyter notebooks)"""
    df = create_results_table(results)
    
    if df.empty:
        print("No results to display.")
        return None
    
    # Get model names for styling
    model_names = results[0].get('model_names', []) if results else []
    
    # Build format dictionary for all numeric columns
    format_dict = {}
    for model_name in model_names:
        format_dict[f'Field Acc ({model_name})'] = '{:.3f}'
        format_dict[f'Line Item ({model_name})'] = '{:.3f}'
        format_dict[f'Financial ({model_name})'] = '{:.3f}'
        format_dict[f'Overall ({model_name})'] = '{:.3f}'
    
    # Create color mapping for winners
    def highlight_winner(row):
        winner = row.get('Winner', '')
        colors = []
        for i in range(len(row)):
            if i == 0:  # Image column
                colors.append('')
            elif row.index[i] == 'Winner':  # Winner column
                # Color based on winner
                if winner in model_names:
                    model_idx = model_names.index(winner)
                    # Use different colors for different models
                    color_map = ['lightgreen', 'lightcoral', 'lightblue', 'lightyellow', 'lightpink']
                    colors.append(f'background-color: {color_map[model_idx % len(color_map)]}')
                else:
                    colors.append('background-color: lightgray')
            else:
                colors.append('')
        return colors
    
    # Apply styling
    styled_df = df.style.set_properties(**{
        'background-color': 'lightblue',
        'color': 'black',
        'border-color': 'white',
        'border-style': 'solid',
        'border-width': '1px'
    }).format(format_dict)
    
    # Highlight winner row
    if 'Winner' in df.columns:
        styled_df = styled_df.apply(
            lambda x: [
                'background-color: lightgreen' if x['Winner'] == model_names[0] 
                else ('background-color: lightcoral' if len(model_names) > 1 and x['Winner'] == model_names[1]
                else ('background-color: lightblue' if len(model_names) > 2 and x['Winner'] == model_names[2]
                else ('background-color: lightyellow' if len(model_names) > 3 and x['Winner'] == model_names[3]
                else 'background-color: lightgray')))
                for i in range(len(x))
            ], 
            axis=1
        )
    
    return styled_df


# Finally displaying the results
display_formatted_table(results)

,Image,Winner,Field Acc (DeepSeek OCR),Line Item (DeepSeek OCR),Financial (DeepSeek OCR),Overall (DeepSeek OCR),Field Acc (Qwen3-VL),Line Item (Qwen3-VL),Financial (Qwen3-VL),Overall (Qwen3-VL),Field Acc (Granite Docling),Line Item (Granite Docling),Financial (Granite Docling),Overall (Granite Docling),Field Acc (Chandra OCR),Line Item (Chandra OCR),Financial (Chandra OCR),Overall (Chandra OCR)
0,invoice_0.png,Qwen3-VL,0.462,0.985,0.998,0.815,0.950,1.000,1.000,0.983,0.815,0.998,1.000,0.938,0.720,0.662,0.944,0.776
1,invoice_1.png,Qwen3-VL,0.450,0.999,1.000,0.816,0.992,0.992,1.000,0.995,0.898,0.956,1.000,0.951,0.744,0.475,0.678,0.632
2,invoice_2.png,Granite Docling,0.485,0.978,1.000,0.821,0.895,0.973,0.998,0.956,0.898,1.000,1.000,0.966,0.664,0.703,0.912,0.760
3,invoice_3.png,Qwen3-VL,0.490,0.996,1.000,0.829,0.988,0.992,1.000,0.993,0.905,0.905,0.992,0.934,0.735,0.690,0.994,0.806
4,invoice_4.png,Qwen3-VL,0.411,0.898,1.000,0.769,0.903,0.944,0.997,0.948,0.862,0.962,0.997,0.941,0.701,0.725,0.942,0.789
5,invoice_5.png,Qwen3-VL,0.488,0.998,1.000,0.829,0.956,1.000,1.000,0.985,0.900,1.000,1.000,0.967,0.827,0.910,0.997,0.911
6,invoice_6.png,Qwen3-VL,0.495,0.906,0.999,0.800,0.900,0.995,0.973,0.956,0.892,0.927,1.000,0.940,0.742,0.658,0.973,0.791
7,invoice_7.png,Qwen3-VL,0.456,0.901,1.000,0.786,0.900,0.962,0.998,0.953,0.856,0.904,0.643,0.801,0.773,0.745,0.956,0.825
8,invoice_8.png,DeepSeek OCR,0.913,0.932,0.973,0.940,0.832,1.000,0.950,0.927,0.878,0.895,0.888,0.887,0.856,0.542,0.761,0.720
9,invoice_9.png,Granite Docling,0.692,0.860,1.000,0.851,0.799,0.835,0.839,0.824,0.769,0.927,1.000,0.899,0.770,0.717,0.988,0.825
